# ⭐ HGNN-ATT-TD Model Training
## Heterogeneous Graph Neural Network with Attention and Temporal Decay

This notebook trains the flagship HGNN-ATT-TD model.

**Prerequisites**: Run `01_preprocessing.ipynb` first to generate preprocessed data.

In [1]:
import sys, os
import warnings
warnings.filterwarnings('ignore')
import pickle

# Ensure PROJECT_P is on the path
PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Project modules
from src.config import *
from src.utils import get_logger, set_plot_style

logger = get_logger('HGNN')
print(f'PyTorch version: {torch.__version__}')
print(f'Device: {DEVICE}')
print('✅ Imports successful')

PyTorch version: 2.11.0+cpu
Device: cpu
✅ Imports successful


In [2]:
# Load preprocessed data from file
preprocess_file = os.path.join(MODEL_DIR, 'preprocessed_data.pkl')

# Check if hetero_data is missing and regenerate if needed
if os.path.exists(preprocess_file):
    with open(preprocess_file, 'rb') as f:
        _check = pickle.load(f)
    if 'hetero_data' not in _check:
        print("⚠️  Preprocessed file missing hetero_data. Regenerating with heterogeneous graph...")
        os.environ["BUILD_HETERO_GRAPH"] = "1"
        from src.data_loader import load_raw_data
        from src.preprocessing import run_preprocessing_pipeline
        
        # Load raw data (10% sample for faster iteration)
        df_raw = load_raw_data(sample_frac=0.1)
        
        # Run preprocessing with hetero_data enabled
        result = run_preprocessing_pipeline(df_raw)
        
        # Save with hetero_data included
        with open(preprocess_file, 'wb') as f:
            pickle.dump({
                'X_train': result['X_train'],
                'X_val': result['X_val'],
                'X_test': result['X_test'],
                'y_train': result['y_train'],
                'y_val': result['y_val'],
                'y_test': result['y_test'],
                'scaler': result['scaler'],
                'label_encoders': result['label_encoders'],
                'feature_names': result['feature_names'],
                'hetero_data': result['hetero_data'],
            }, f)
        print(f"✅ Regenerated {preprocess_file} with hetero_data")

if not os.path.exists(preprocess_file):
    raise FileNotFoundError(f'Please run preprocessing first to generate {preprocess_file}')

# Now load the data (which should have hetero_data)
with open(preprocess_file, 'rb') as f:
    prep = pickle.load(f)

X_train = prep['X_train']
X_val = prep['X_val']
X_test = prep['X_test']
y_train = prep['y_train']
y_val = prep['y_val']
y_test = prep['y_test']

# Check if hetero_data is available
if 'hetero_data' not in prep:
    raise KeyError(
        '❌ hetero_data not found in preprocessed data.\n'
        'The saved pickle does not include the heterogeneous graph.\n'
        'Regenerate by running: src/preprocessing.py with BUILD_HETERO_GRAPH=1\n'
        '  export BUILD_HETERO_GRAPH=1 (Linux/Mac)\n'
        '  set BUILD_HETERO_GRAPH=1 (Windows)\n'
        'Then re-run preprocessing with: python src/preprocessing.py'
    )

hetero_data = prep['hetero_data']

print(f'✅ Preprocessed data loaded')
print(f'   Train: {X_train.shape}')
print(f'   Val:   {X_val.shape}')
print(f'   Test:  {X_test.shape}')
if hetero_data is not None:
    print(f'\nHetero Graph:')
    print(f'  Nodes: {hetero_data.node_types}')
    print(f'  Edges: {hetero_data.edge_types}')
    print(f'  Transactions: {hetero_data["transaction"].x.shape}')


✅ Preprocessed data loaded
   Train: (41337, 217)
   Val:   (8858, 217)
   Test:  (8859, 217)

Hetero Graph:
  Nodes: ['transaction', 'card', 'device']
  Edges: [('transaction', 'used', 'card'), ('card', 'used_by', 'transaction'), ('transaction', 'on', 'device'), ('device', 'hosts', 'transaction')]
  Transactions: torch.Size([59054, 217])


2. Train HGNN-ATT-TD

In [3]:
from src.training import train_neural_network

hgnn_model, hgnn_history = train_neural_network(
    X_train, y_train, X_val, y_val,
    hetero_data=hetero_data,
    use_smote=False,  # FocalLoss handles imbalance
)
print('\n✅ HGNN-ATT-TD training complete')

17:57:02 | Training             | INFO    | ==================================================
17:57:02 | Training             | INFO    |   🧠 Training Neural Network (HGNN-ATT-TD)
17:57:02 | Training             | INFO    | ==================================================
17:57:02 | Training             | INFO    |   Device: cpu (MPS not supported by PyG HGTConv)
17:57:02 | Models               | INFO    |   🧠 FraudHGNN created (146,903 parameters)
17:57:02 | Models               | INFO    |      Architecture: HGTConv × 2 layers, 2 attention heads
17:57:02 | Models               | INFO    |      Loss: FocalLoss(γ=2.0, α=0.75)
17:57:02 | Training             | INFO    | ⏳ Starting: HGNN-ATT-TD training (full-batch)
17:57:05 | Training             | INFO    |   Epoch   1/30 | Loss: 0.0571 | Val AUC: 0.5776
17:57:16 | Training             | INFO    |   Epoch   5/30 | Loss: 0.0354 | Val AUC: 0.6969
17:57:28 | Training             | INFO    |   Epoch  10/30 | Loss: 0.0208 | Val AUC: 0.80


✅ HGNN-ATT-TD training complete


3. Evaluate on Test Set

In [4]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# PyG HGTConv does NOT support MPS. Force CPU for inference.
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
hgnn_device = torch.device("cpu")

# Predictions
hgnn_model = hgnn_model.to(hgnn_device)  # Move model to CPU for inference
hgnn_model.eval()
with torch.no_grad():
    hetero_data = hetero_data.to(hgnn_device)

    # Move all node feature tensors to CPU
    x_dict = {node_type: x.to(hgnn_device) for node_type, x in hetero_data.x_dict.items()}
    edge_index_dict = {edge_type: edge.to(hgnn_device) for edge_type, edge in hetero_data.edge_index_dict.items()}

    # Get temporal decay if available
    tx_decay = getattr(hetero_data['transaction'], 'time_decay', None)
    if tx_decay is not None:
        tx_decay = tx_decay.to(hgnn_device)

    # Forward pass on full heterogeneous graph
    logits_full = hgnn_model(
        x_dict,
        edge_index_dict,
        tx_time_decay=tx_decay,
    )

    # Extract test predictions (using test_mask if available, otherwise use all)
    if hasattr(hetero_data['transaction'], 'test_mask'):
        logits_test = logits_full[hetero_data['transaction'].test_mask]
    else:
        # Fallback: use the last len(y_test) samples
        logits_test = logits_full[-len(y_test):]

    y_pred_proba = torch.sigmoid(logits_test).cpu().numpy().flatten()

y_pred = (y_pred_proba > 0.5).astype(int)

# Metrics
metrics = {
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred, zero_division=0),
    'Recall': recall_score(y_test, y_pred, zero_division=0),
    'F1-Score': f1_score(y_test, y_pred, zero_division=0),
    'ROC-AUC': roc_auc_score(y_test, y_pred_proba),
}

print('\n' + '='*50)
print('HGNN-ATT-TD - Test Set Performance')
print('='*50)
for metric, value in metrics.items():
    print(f'{metric:15s}: {value:.4f}')
print('='*50)


HGNN-ATT-TD - Test Set Performance
Accuracy       : 0.9646
Precision      : 0.5083
Recall         : 0.1930
F1-Score       : 0.2798
ROC-AUC        : 0.8163


4. Plot Training History

In [5]:
set_plot_style()

if hgnn_history:
    print(f"Available keys in hgnn_history: {list(hgnn_history.keys())}")

    # Plot what we have
    if 'train_losses' in hgnn_history and 'val_aucs' in hgnn_history:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # Loss curves
        axes[0].plot(hgnn_history['train_losses'], label='Train Loss', linewidth=2, color='#1f77b4')
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('Loss')
        axes[0].set_title('Training History - Loss')
        axes[0].legend()
        axes[0].grid(alpha=0.3)

        # Validation AUC
        axes[1].plot(hgnn_history['val_aucs'], label='Validation AUC', linewidth=2, color='#ff7f0e')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('AUC Score')
        axes[1].set_title('Training History - Validation AUC')
        axes[1].legend()
        axes[1].grid(alpha=0.3)

        plt.tight_layout()
        plt.savefig(os.path.join(EVAL_DIR, '06_hgnn_training_curves.png'), dpi=150, bbox_inches='tight')
        plt.show()
        print('✅ Training curves saved')
    else:
        print('⚠️ Unexpected training history structure')
else:
    print('⚠️ No training history available')

Available keys in hgnn_history: ['train_losses', 'val_aucs']
✅ Training curves saved


5. Save Model and Results

In [6]:
# Save model results (predictions only, not full model to avoid serialization issues)
hgnn_results = {
    'y_pred': y_pred,
    'y_pred_proba': y_pred_proba,
    'metrics': metrics,
    'history': hgnn_history,
}

results_file = os.path.join(MODEL_DIR, 'hgnn_results.pkl')
with open(results_file, 'wb') as f:
    pickle.dump(hgnn_results, f)

print(f'✅ Results saved to {results_file}')
print('\n📝 Next: Run 05_evaluation.ipynb to compare all models')

✅ Results saved to c:\Users\Lekshmi Priya\OneDrive\Documents\GitHub\Credit-Card-Fraud-Detection-System\models\hgnn_results.pkl

📝 Next: Run 05_evaluation.ipynb to compare all models
